# 3. embedding·pgvector·retrieval 계약

**시나리오:** 휴가 신청 시점을 묻는 질문과 무관한 Wi-Fi 질문을 구분합니다.

**학습 목표:** fixture retriever로 검색 계약을 관찰하고, live 모드의 `OpenAIEmbeddings` → `PGVector` → similarity search 연결을 이해합니다.

## 중요 변수·함수

- `POLICY_DOCUMENTS`: 실제 저장·검색되는 chunk입니다.
- `create_fixture_services().retriever`: 비용 없이 검색 결과를 재현합니다.
- live 모드에서는 `build_live_adapters()`가 `OpenAIEmbeddings`와 `PGVector`를 만들며 같은 `search()` 계약을 사용합니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# 관련 질문은 근거 chunk를 반환합니다.
from week1.app import POLICY_DOCUMENTS, create_fixture_services

services = create_fixture_services()
relevant = services.retriever.search('vacation request three business days')
[(doc.metadata['chunk_id'], doc.page_content) for doc in relevant]

In [ ]:
# 무관한 질문은 빈 결과여야 hallucination을 막을 수 있습니다.
unrelated = services.retriever.search('office wifi password')
assert unrelated == []
{'relevant_count': len(relevant), 'unrelated_count': len(unrelated)}

## 예측 과제와 해석

**예측 과제:** 검색 결과가 비어 있는데 모델을 호출하면 어떤 위험이 생길지 설명하세요.

**해석:** 검색기는 답변기가 아니라 근거 선택기입니다. fixture와 live 구현은 내부 기술이 달라도 빈 근거를 같은 방식으로 표현해야 workflow가 안전하게 중단됩니다.